# Microglia-Inspired Dynamic Pruning — Qwen3-4B Evaluation

Train small 'agent' networks to learn which attention heads to prune,
then evaluate on GSM8K math reasoning.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/microglia-pruning/blob/main/notebooks/qwen3_pruning_eval.ipynb)


## Setup

In [ ]:
# Remove any existing clone and get fresh copy
import os
import shutil

if os.path.exists('/content/microglia-pruning'):
    shutil.rmtree('/content/microglia-pruning')
    print('Removed old clone')

!git clone https://github.com/Tommaso-R-Marena/microglia-pruning.git
%cd microglia-pruning
!git log --oneline -1


In [ ]:
!pip install -q torch transformers accelerate bitsandbytes peft datasets scipy numpy tqdm matplotlib
print('✓ Installation complete')


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import sys
import time

sys.path.insert(0, '/content/microglia-pruning')
from src.system import MicrogliaPruningSystem

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch.manual_seed(42)
np.random.seed(42)


## Part 1: Load Model

Using `Qwen/Qwen3-4B-Instruct`. First run downloads weights (~3–5 min).


In [ ]:
# Initialize the pruning system
# This loads Qwen/Qwen3-4B-Instruct and creates pruning agents for each layer

system = MicrogliaPruningSystem(
    model='Qwen/Qwen3-4B-Instruct',   # fix: removed nonexistent -2507 suffix
    num_heads=32,                      # Qwen3-4B has 32 Q-attention heads per layer
    hidden_dim=128,
    temperature=1.0
)

print('\n✓ System initialized!')
print(f'Model size: {sum(p.numel() for p in system.model.parameters())/1e9:.2f}B parameters')
print(f'Agent size: {sum(p.numel() for p in system.agents.parameters())/1e6:.2f}M parameters')
print(f'Agent overhead: {sum(p.numel() for p in system.agents.parameters())/sum(p.numel() for p in system.model.parameters())*100:.3f}%')


## Part 2: Baseline Spot-Check

Quick sanity-check on 3 examples *before* training.


In [ ]:
# Test on a couple examples before training
test_questions = [
    'A store sells apples for $2 each. If Sarah buys 5 apples, how much does she spend?',
    'John has 15 candies. He gives 3 to each of his 4 friends. How many candies does he have left?',
    'A rectangle has a length of 8 meters and a width of 5 meters. What is its area?'
]

print('Testing baseline (unpruned) model:\n')
system._enable_pruning(False)

for i, question in enumerate(test_questions, 1):
    # Use _format_prompt so the Qwen3-Instruct chat template is applied correctly.
    # Passing a raw 'Question: ...\nAnswer:' string to an instruct model yields incoherent output.
    prompt = system._format_prompt(question)

    start = time.time()
    output = system.generate(prompt, max_new_tokens=128)
    elapsed = time.time() - start

    answer = system._extract_answer(output)
    print(f'Q{i}: {question}')
    print(f'Raw output: {output[:200]}')
    print(f'Extracted answer: {answer}')
    print(f'Time: {elapsed:.2f}s\n')


## Part 3: Train Pruning Agents

Curriculum learning — sparsity pressure increases from α=0.01 to α=0.2.
**~15–20 min on T4.**


In [ ]:
# Train the pruning agents with curriculum sparsity schedule
system.train(
    dataset_name='gsm8k',
    num_epochs=3,
    batch_size=2,
    learning_rate=1e-4,
    alpha_schedule=(0.01, 0.2),   # sparsity pressure rises linearly each epoch
    use_lora=False,               # LoRA wraps attention *after* pruning hooks; keep off
    precision='bf16',             # bfloat16 is numerically stable on T4/A100; avoids fp16 overflow
    max_length=512,               # 256 was truncating ~19% of GSM8K examples
    max_steps_per_epoch=None,     # run the full epoch rather than stopping at 30 steps
    max_val_samples=50,           # use 50 examples for early-stopping validation
)

print('\n✓ Training complete!')


## Part 4: Training Curves


In [ ]:
import math
history = system.training_history

# Filter out any NaN epochs (can happen if the label-masking fix wasn't applied)
history = [h for h in history if not math.isnan(h.get('task_loss', float('nan')))]

if history:
    epochs = range(1, len(history) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].plot(epochs, [h['task_loss']    for h in history], 'b-o', lw=2, ms=8)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Task Loss')
    axes[0].set_title('Task Performance'); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, [h['sparsity_loss'] for h in history], 'r-o', lw=2, ms=8)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Sparsity Loss')
    axes[1].set_title('Pruning Pressure'); axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs, [h['total_loss']   for h in history], 'g-o', lw=2, ms=8)
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Total Loss')
    axes[2].set_title('Total Loss'); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('No valid training history — re-run the training cell.')


## Part 5: Evaluate Pruned Model

200 test examples from GSM8K (~28 min on T4).


In [ ]:
results = system.evaluate(
    dataset_name='gsm8k',
    split='test',
    max_samples=200
)

ci = results['ci_95']   # tuple (lower, upper)
print('\n' + '='*50)
print('FINAL RESULTS')
print('='*50)
print(f"Accuracy:  {results['accuracy']:.1%}  [95% CI: {ci[0]:.1%} – {ci[1]:.1%}]")
print(f"Correct:   {results['correct']}/{results['total']}")
print(f"Sparsity:  {results['sparsity']:.1%} heads pruned")
print('='*50)


## Part 6: Pruning Heatmap


In [ ]:
# Run one forward pass to populate last_masks on each wrapped attention layer
_ = system.generate(
    system._format_prompt('What is 2 + 2?'),
    max_new_tokens=50,
    use_pruning=True
)

# Collect masks using the safe accessor pattern (handles self_attn vs attn naming)
all_masks = []
for layer in system.get_layers():
    attn = getattr(layer, 'self_attn', getattr(layer, 'attn', None))
    if attn is not None and hasattr(attn, 'last_masks') and attn.last_masks is not None:
        avg_mask = attn.last_masks.mean(dim=0).cpu().numpy()
        all_masks.append(avg_mask)

if all_masks:
    mask_matrix = np.array(all_masks)
    plt.figure(figsize=(12, 8))
    im = plt.imshow(mask_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    plt.colorbar(im, label='Keep Probability')
    plt.xlabel('Head Index', fontsize=12)
    plt.ylabel('Layer Index', fontsize=12)
    plt.title('Pruning Pattern Across Layers', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f'Overall sparsity: {1 - mask_matrix.mean():.1%}')
else:
    print('No masks captured — ensure pruning is enabled and agents are trained.')


## Summary

1. Loaded `Qwen/Qwen3-4B-Instruct`  
2. Trained small pruning agents (3 epochs, curriculum α schedule)  
3. Evaluated on 200 GSM8K test problems with bootstrap CIs  
4. Visualised per-layer, per-head pruning patterns  

**Issues or questions?** → [github.com/Tommaso-R-Marena/microglia-pruning](https://github.com/Tommaso-R-Marena/microglia-pruning)
